# Build ecmtool inputs

Prepares the Rhodoferax and Geobacter models for ecmtool: for each species and each exchange pattern, fixes a growth rate and carbon-uptake rate, converts the model into a bound-free form ecmtool can read, and writes out one SBML file per species/pattern/growth-rate combination plus the shell script to run ecmtool on all of them.

1. Load the model, restrict exchanges to the pattern's allowed uptakes/secretions, close all others.
2. Determine the maximum growth rate under this pattern.
3. For a set of growth-rate fractions (0.3-0.99 of the max), fix growth rate and carbon uptake:
    * Add a `fixed_growth` metabolite, produced by the biomass reaction.
    * Add a `fixed_uptake` metabolite, consumed by the carbon-source uptake reaction.
    * Add `constraint_reaction`, with flux fixed at 1, consuming both metabolites at rates that encode the target growth rate and uptake. This converts growth rate and uptake into equalities, since ecmtool cannot read bounds from the SBML file - it treats every row as an equality.
    * Add `slack_reaction` so the uptake constraint stays an inequality (uptake <= `FIXED_UPTAKE`) instead of becoming an equality too.
4. Export one SBML file per species/pattern/growth-rate combination.
5. Find the `--tag` index ecmtool needs for `constraint_reaction`, and write the `.sh` file that runs ecmtool on all generated models.
6. Save everything a downstream notebook needs (growth rates, patterns, tags) to `run_metadata.pkl`, so it doesn't have to be copy-pasted.

In [1]:
import os
import sys
import pickle
import contextlib
from collections import defaultdict

import cobras
import numpy as np
from cobra import Reaction
from helper import flip_reverse_reactions, add_product_to_reaction

sys.path.insert(0, "/home/users/dszeliova/ecmtool")
from ecmtool.network import extract_sbml_stoichiometry

cobra_config = cobra.Configuration()
cobra_config.solver = "cplex"
cobra_config.tolerance = 1e-9

# Maximum carbon-source uptake rate enforced everywhere below 
# Defined once here and saved to run_metadata.pkl
# at the end, so analyze_ecms.ipynb reads it instead of hardcoding it again.
FIXED_UPTAKE = 10

/home/users/dszeliova/ecmtool/ecmtool/_bglu_dense.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import sys, pkg_resources, imp


### Define exchange patterns

Selected exchange patterns from Oftadeh et al. 2024: https://doi.org/10.1101/2024.01.30.577913

In [2]:
bm_rxns = {"Geobacter": "agg_GS13m", "Rhodoferax": "BIO_Rfer3"}
c_sources = ["EX_ac_e", "EX_mal-L_e", "EX_cit_e", "EX_fum_e", "EX_lac-L_e"]

exchange_pattern = {"Rhodoferax": {
            "mEmC_1": {"uptakes": ["EX_fe2_e", "EX_fum_e"],
                        "secretion": ["EX_mal-L_e"]},
            "mEmC_3": {"uptakes": [ "EX_fe2_e", "EX_fum_e"],
                        "secretion": ["EX_mal-L_e"]},
            "mEmC_2": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
                      "secretion": ["EX_mal-L_e"]},
            "mEmC_4": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
                        "secretion": ["EX_mal-L_e"]},
            # "mEmC_5": {"uptakes": [ "EX_fe2_e", "EX_mal-L_e"],
            #             "secretion": []},
            "mE_1": {"uptakes": ["EX_ac_e", "EX_fe3_e"],
                    "secretion": []},
            "mU_1": {"uptakes": ["EX_ac_e", "EX_fe3_e"],
                     "secretion": []}
},
        "Geobacter": {
            "mEmC_1": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
                      "secretion": []},
            "mEmC_2": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
                      "secretion": []},
            "mEmC_3": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                      "secretion": []},
            "mEmC_4": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                      "secretion": []},
            # "mEmC_5": {"uptakes": ["EX_n2_e", "EX_cit_e"],
            #           "secretion": ["EX_mal-L_e"]},
            "mE_1": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                  "secretion": ["EX_ac_e"]},
            "mU_1": {"uptakes": ["EX_lac-L_e", "EX_n2_e"],
                     "secretion": ["EX_ac_e"]},
    }}

def open_exchanges(model, organism):

    if organism == 'Geobacter':
        allowed = ['so4_e', 'pi_e', 'mg2_e', 'k_e', 'ca2_e', 'fe3_e']
    elif organism == 'Rhodoferax':
        allowed = ['nh4_e', 'pi_e', 'so4_e', 'o2_e']
    else:
        raise ValueError(f"Unknown organism: {organism}")

    # remove the demand reactions in Geobacter
    if organism == "Geobacter":
        model.reactions.ATPM.bounds = (0., 1000.)

        to_remove = [rxn.id for rxn in model.reactions if rxn.id.startswith("DM_")]
        model.remove_reactions(to_remove)

    return allowed


def implement_pattern(model, allowed, pattern, c_sources):
    # first, allow all exchanges as in the paper and turn off all secretions
    mu = model.slim_optimize()
    print(f" * Initial growth rate: {mu:.2f}")

    for ex in model.exchanges:
        if ex.id in [ "EX_h2o_e",  "EX_h2_e",  "EX_h_e" ]:
            ex.bounds = (-1000, 1000)
            continue

        # pattern should overwrite the allowed
        if ex.id in pattern["secretion"]:
            ex.bounds = (0, 1000)
        elif (ex.id.replace("EX_", "") in allowed) or (ex.id in pattern["uptakes"]):
            ex.bounds = (-1000, 0)
            if ex.id in c_sources:
                ex.bounds = (-FIXED_UPTAKE, 0)
        else:
            ex.bounds = (0, 1000)

    mu = model.slim_optimize()
    print(f" * Growth rate with exchange pattern: {mu:.2f}")

    return mu

### Implement exchange pattern

Define uptakes according to the pattern and remove reactions with lower and upper bounds of zero. Test growth after implementing the constraints.

In [3]:
reduced_models = {"Rhodoferax": {}, "Geobacter": {}}
pattern_max_growth = {}
for modelname in reduced_models:
    print(f"\n***** {modelname} *****")

    for pattern_name, pattern in exchange_pattern[modelname].items():
        print(f"Testing exchange pattern {pattern_name}")

        model = cobra.io.load_matlab_model(f"../models/{modelname}.mat")
        allowed = open_exchanges(model, modelname)

        to_remove = []
        for rxn in model.reactions:
            if abs(rxn.lower_bound) < 1e-15 and abs(rxn.upper_bound) < 1e-15:
                print(f"{rxn.id} removed\n")
                to_remove.append(rxn)
        model.remove_reactions(to_remove)
        cobra.manipulation.delete.prune_unused_metabolites(model)

        # needed for pycomo
        if pattern_name == "mEmC_1":
            cobra.io.write_sbml_model(model, f"../models/xml_models/{modelname}.xml")
        
        mu = implement_pattern(model, allowed, pattern, c_sources)

        pattern_max_growth.setdefault(pattern_name, []).append(mu)

        reduced_models[modelname][pattern_name] = model


***** Rhodoferax *****
Testing exchange pattern mEmC_1


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


 * Initial growth rate: 41.92
 * Growth rate with exchange pattern: 0.67
Testing exchange pattern mEmC_3


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


 * Initial growth rate: 41.92
 * Growth rate with exchange pattern: 0.67
Testing exchange pattern mEmC_2


...

FUMt4 removed

 * Initial growth rate: 35.65
 * Growth rate with exchange pattern: 0.93
Testing exchange pattern mE_1


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
/home/users/dszeliova/.conda/envs/comms2/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


FUMt4 removed

 * Initial growth rate: 35.65
 * Growth rate with exchange pattern: 0.93
Testing exchange pattern mU_1


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
/home/users/dszeliova/.conda/envs/comms2/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


FUMt4 removed

 * Initial growth rate: 35.65
 * Growth rate with exchange pattern: 0.76


In [4]:
pattern_max_growth

{'mEmC_1': [0.6698781291076985, 1.015500608059892],
 'mEmC_3': [0.6698781291076985, 0.9311962701878479],
 'mEmC_2': [1.0048171936615455, 1.015500608059892],
 'mEmC_4': [1.0048171936615455, 0.9311962701878479],
 'mE_1': [0.3694464604864135, 0.9311962701878479],
 'mU_1': [0.3694464604864135, 0.7616254560449183]}

### Convert models to bound-free form

Fix growth rate and max. uptake rate by adding artificial reactions and metabolites directly to the SBML (ecmtool cannot read cobra lower and upper bounds).

In [5]:
new_names = []
simulated_mus = {}
for modelname in reduced_models:
    print(f"\n***** {modelname} *****")
    for pattern_name, model in reduced_models[modelname].items():
        print(pattern_name)
        simulated_mus[pattern_name] = []

        pattern = exchange_pattern[modelname][pattern_name]
        min_mu = min(pattern_max_growth[pattern_name])  # lower growth rate is limiting

        for fraction in [0.99, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]:
            model_modified = model.copy()
            set_mu = min_mu * fraction
            simulated_mus[pattern_name].append(set_mu)

            # modify model such that constraints are part of matrix
            add_product_to_reaction(model_modified,
                                    reaction_id=bm_rxns[modelname],
                                    metabolite_id="fixed_growth",
                                    stoichiometry=1.0,
                                    name="fixed_growth")

            c_source = sorted(set(pattern["uptakes"]) & set(c_sources))
            if len(c_source) != 1:
                raise ValueError(
                    f"Expected exactly one carbon source in {modelname}/{pattern_name}, got {c_source}"
                )

            model_modified.reactions.get_by_id(c_source[0]).lower_bound = -1000

            add_product_to_reaction(model_modified,
                                    reaction_id=c_source[0],
                                    metabolite_id="fixed_uptake",
                                    stoichiometry=-1.0,
                                    name="fixed_uptake")

            # reaction with values for the constraints
            # when flux 1 - constraints are active
            constraint_reaction = Reaction("constraint_reaction")
            constraint_reaction.lower_bound = 1
            constraint_reaction.upper_bound = 1
            constraint_reaction.add_metabolites({
                model_modified.metabolites.fixed_growth: -set_mu,
                model_modified.metabolites.fixed_uptake: -FIXED_UPTAKE
            })

            # slack variable to allow inequality constraint for uptake
            slack_reaction = Reaction("slack_reaction")
            slack_reaction.lower_bound = 0
            slack_reaction.upper_bound = 1000
            slack_reaction.add_metabolites({
                model_modified.metabolites.fixed_uptake: 1
            })
            model_modified.add_reactions([constraint_reaction, slack_reaction])

            flip_reverse_reactions(model_modified)

            mu = model_modified.slim_optimize()
            print(f" * Growth rate after preparation for ecmtool {mu:.2f}")

            new_name = f"{modelname}_{pattern_name}_{round(set_mu, 2)}.xml"
            cobra.io.write_sbml_model(model_modified, f"../models/{new_name}")
            new_names.append(new_name)


***** Rhodoferax *****
mEmC_1
 * Growth rate after preparation for ecmtool 0.66
 * Growth rate after preparation for ecmtool 0.60
 * Growth rate after preparation for ecmtool 0.54
 * Growth rate after preparation for ecmtool 0.47
 * Growth rate after preparation for ecmtool 0.40
 * Growth rate after preparation for ecmtool 0.33
 * Growth rate after preparation for ecmtool 0.27
 * Growth rate after preparation for ecmtool 0.20
mEmC_3
 * Growth rate after preparation for ecmtool 0.66
 * Growth rate after preparation for ecmtool 0.60
 * Growth rate after preparation for ecmtool 0.54
 * Growth rate after preparation for ecmtool 0.47
 * Growth rate after preparation for ecmtool 0.40
 * Growth rate after preparation for ecmtool 0.33
 * Growth rate after preparation for ecmtool 0.27
 * Growth rate after preparation for ecmtool 0.20
mEmC_2
 * Growth rate after preparation for ecmtool 0.99
 * Growth rate after preparation for ecmtool 0.90
 * Growth rate after preparation for ecmtool 0.80
 * Gr

### Create input for ecmtool

Generates the `.sh` file that runs ecmtool from the command line on every model written above.

By default, ecmtool only reports exchanges.
We need ecmtool to report the flux of `constraint_reaction` in its output, so we can normalize each ECM by it. ecmtool exposes this through the `--tag` flag, but it takes a reaction *index*, not a reaction ID - and that index is not the reaction's position in the SBML file. Before assigning indices, ecmtool drops every reaction that touches only one metabolite (its own definition of an exchange reaction), so the index has to be found after that filtering step, not read off the SBML file directly. `find_tag_index()` (below) computes this automatically by running ecmtool's own model-parsing step and reading off the position of `constraint_reaction` in the resulting reaction list.

In [6]:
def find_tag_index(model_path, reaction_id, add_objective=True):
    """Return the ecmtool --tag index for reaction_id in model_path.

    ecmtool drops every reaction touching only one metabolite before
    assigning indices, so this mirrors that step instead of guessing.
    """
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        network = extract_sbml_stoichiometry(
            model_path, add_objective=add_objective, skip_external_reactions=True
        )
    for index, rxn in enumerate(network.reactions):
        if rxn.id in (reaction_id, f"R_{reaction_id}"):
            return index
    raise ValueError(f"'{reaction_id}' not found or was dropped as dead-end.")

`constraint_reaction` is added identically for every growth rate of a given organism, so its tag index should be the same across all of that organism's models - the assertion below checks this instead of assuming it.

In [7]:
def get_organism(name):
    return name.split("_")[0]

tags_by_organism = defaultdict(set)
for name in new_names:
    tag = find_tag_index(f"../models/{name}", "constraint_reaction")
    tags_by_organism[get_organism(name)].add(tag)

for organism, found_tags in tags_by_organism.items():
    assert len(found_tags) == 1, f"{organism} has inconsistent tags: {found_tags}"

tags = {organism: next(iter(found_tags)) for organism, found_tags in tags_by_organism.items()}
tags

{'Rhodoferax': 761, 'Geobacter': 523}

In [8]:
def write_run_script(new_names, tags, path="run_models_ecmtool.sh"):
    models_by_organism = defaultdict(list)
    for name in new_names:
        models_by_organism[get_organism(name)].append(name)

    lines = [
        "#!/bin/bash",
        "# IMPORTANT: Run this script from the ecmtool directory (ecmtool) where main.py is located",
        "# activate comms2 environment",
        "# inputs created with make_ecmtool_inputs.ipynb",
        "# --tag: index of constraint_reaction, determined with make_ecmtool_inputs.ipynb (find_tag_index)",
        "",
    ]
    for organism, names in models_by_organism.items():
        lines.append("models=(")
        lines += [f'"{name}"' for name in names]
        lines.append(")")
        lines += [
            'for model in "${models[@]}"; do',
            '  out="../uranium/results/${model%.xml}_full_conversions.csv"',
            '  if [ -f "$out" ]; then',
            '    echo "Skipping ${model}: output already exists"',
            "  else",
            "    python main.py \\",
            '      --model_path "../uranium/models/${model}" \\',
            "      --compress true \\",
            "      --remove_infeasible false \\",
            f"      --tag {tags[organism]} \\",
            '      --out_path "$out"',
            "  fi",
            "done",
        ]

    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")

write_run_script(new_names, tags)

### Save run metadata

Everything `analyze_ecms.ipynb` needs to interpret the ecmtool output - growth rates tested, the exchange patterns, the fixed uptake rate.

In [9]:
run_metadata = {
    "simulated_mus": simulated_mus,
    "fixed_uptake": FIXED_UPTAKE,
    "c_sources": c_sources,
    "exchange_pattern": exchange_pattern
}
with open("../run_metadata.pkl", "wb") as f:
    pickle.dump(run_metadata, f)